### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos el archivo language.csv de nuestro contenedor bronze

In [0]:
df = spark.read.option("header", "true")\
          .option("inferSchema", "true")\
          .csv(f"{bronze_folder_path}/language.csv")

###### Seleccionamos solo las columnas que nos interesan

In [0]:
from pyspark.sql.functions import col,current_timestamp, lit
df_consulta = df.select(col("languageId"), col("languageName"))

###### Cambiamos el nombre de las columnas

In [0]:
df_renamed = df_consulta.withColumnRenamed("languageId", "language_id")\
                        .withColumnRenamed("languageName", "language_name")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
df_add = add_columnas_control(df_renamed,v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/language")